# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


In [ ]:
# =============================
# STEP 1: Install dependencies
# =============================
# Install once if needed: pip install google-cloud-storage

# =============================
# STEP 2: Authenticate
# =============================

# =============================
# STEP 3: Imports
# =============================
from google.cloud import storage
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# =============================
# STEP 4: Set GCS bucket & folder
# =============================
bucket_name = os.getenv("GCS_BUCKET", "your-gcs-bucket")
prefix = os.getenv("GCS_SOURCE_PREFIX", "path/to/soft-tissue-sarcoma/")

client = storage.Client()
bucket = client.bucket(bucket_name)

# =============================
# STEP 5: List files only in the target folder
# =============================
blobs = client.list_blobs(bucket_name, prefix=prefix)

data = []

for blob in tqdm(blobs, desc="Listing files"):
    if blob.name.endswith('/'):  # Ignore folders
        continue

    # Remove prefix to get relative path
    relative_path = blob.name.replace(prefix, '', 1)

    # Split folder path (excluding filename)
    folder_parts = relative_path.split('/')[:-1]
    filename = relative_path.split('/')[-1]
    size_mb = round(blob.size / (1024 * 1024), 2)

    # Garantir que existam sempre 3 níveis (preenche com None se faltar)
    case_id, study_info, series_info = (folder_parts + [None, None, None])[:3]

    data.append({
        'case_id': case_id,
        'study_info': study_info,
        'series_info': series_info,
        'file_name': filename,
        'size_MB': size_mb
    })

# Criar DataFrame direto com as colunas finais
df = pd.DataFrame(data, columns=['case_id', 'study_info', 'series_info', 'file_name', 'size_MB'])

# =============================
# STEP 6: Display the DataFrame
# =============================
print(f"Total files found: {len(df)}")
df.head(20)

# =============================
# Save to CSV
# =============================
Path("data").mkdir(exist_ok=True)
df.to_csv("data/sts_file_list.csv", index=False)
print("CSV saved to data/sts_file_list.csv")


In [ ]:
# =============================
# STEP 7: Extract unique RTStruct types (case-insensitive sorted)
# =============================

# Filter only rows that contain "RTstruct" (case-insensitive)
rtstruct_df = df[df['series_info'].str.contains('RTstruct', case=False, na=False)]

# Extract only the part between the first and second "-"
rtstruct_types = rtstruct_df['series_info'].str.extract(r'^[^/]*?-(.*?)-')[0]

# Drop NaN values and remove duplicates
rtstruct_unique_list = rtstruct_types.dropna().unique().tolist()

# Sort alphabetically ignoring case but keep original capitalization
rtstruct_unique_list = sorted(rtstruct_unique_list, key=lambda x: x.lower())

# Print the result vertically
print(f"Unique RTStruct types found: {len(rtstruct_unique_list)}\n")
for item in rtstruct_unique_list:
    print(item)


In [ ]:
# =============================
# STEP 9: Extract unique NON-RTStruct series types (fully cleaned)
# =============================

import re

# Filter rows that DO NOT contain "RTstruct" (case-insensitive)
non_rtstruct_df = df[~df['series_info'].str.contains('RTstruct', case=False, na=False)]

# Extract only the part between the first and second "-"
non_rtstruct_types = non_rtstruct_df['series_info'].str.extract(r'^[^/]*?-(.*?)-')[0]

# Drop NaN
clean_series = non_rtstruct_types.dropna()

# Remove leading numbering like "2. ", "3 ", "4) "
clean_series = clean_series.str.replace(r'^\s*\d+[.) ]+\s*', '', regex=True)

# Strip extra spaces
clean_series = clean_series.str.strip()

# Remove entries that are ONLY digits or blank
clean_series = clean_series[~clean_series.str.match(r'^\s*$')]           # blanks
clean_series = clean_series[~clean_series.str.match(r'^\d+\.?$')]        # pure numbers

# Normalize multiple spaces into one
clean_series = clean_series.str.replace(r'\s+', ' ', regex=True)

# Get unique values and sort alphabetically ignoring case
series_unique_list = sorted(clean_series.unique().tolist(), key=lambda x: x.lower())

# Print the cleaned unique list vertically
print(f"Cleaned unique NON-RTStruct series types found: {len(series_unique_list)}\n")
for item in series_unique_list:
    print(item)


In [ ]:
# =============================
# STEP 10: Normalized best-match + keyword fallback + elimination
# =============================

from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def normalize_sequence_name(name):
    """Extract key sequence type keywords for better matching"""
    name = name.lower()
    keywords = ["t1", "t2", "stir", "pet", "ct", "fat", "fse", "ax", "cor", "sag"]
    found = [k for k in keywords if k in name]
    return " ".join(sorted(set(found)))

def extract_main_keyword(name):
    """Return the most important keyword (t1, t2, stir, etc.) if present"""
    name = name.lower()
    for kw in ["t1", "t2", "stir", "pet", "ct"]:
        if kw in name:
            return kw
    return None

folder_mapping = {}
unmatched_count = 0
total_rtstruct_count = 0

for (case_id, study_info), group_df in df.groupby(["case_id", "study_info"]):
    study_series = group_df['series_info'].dropna().unique().tolist()
    rtstruct_series = [s for s in study_series if "RTstruct" in s]
    dicom_series = [s for s in study_series if "RTstruct" not in s]

    used_dicom = set()
    mappings = []
    unmatched_rt = []  # store rtstruct that couldn't match on first pass
    unmatched_dicom = dicom_series.copy()  # will remove matched ones

    # First pass: normalized similarity
    for rt in rtstruct_series:
        total_rtstruct_count += 1
        rt_base = rt.replace("RTStruct", "").replace("RTstruct", "")
        rt_norm = normalize_sequence_name(rt_base)

        best_match = None
        best_score = 0.0

        for dicom in dicom_series:
            if dicom in used_dicom:
                continue

            dicom_base = dicom.split("-", 2)[1] if "-" in dicom else dicom
            dicom_norm = normalize_sequence_name(dicom_base)
            score = similarity(rt_norm, dicom_norm)

            if score > best_score:
                best_score = score
                best_match = dicom

        # Consider it matched only if similarity is meaningful
        if best_match and best_score >= 0.5:
            used_dicom.add(best_match)
            unmatched_dicom.remove(best_match)
            mappings.append({
                "rtstruct_series": rt,
                "matched_dicom_series": best_match,
                "similarity": round(best_score, 3),
                "method": "similarity"
            })
        else:
            unmatched_rt.append(rt)

    # Second pass: keyword-based fallback
    still_unmatched_rt = []
    for rt in unmatched_rt:
        rt_kw = extract_main_keyword(rt)
        found_kw_match = None

        if rt_kw:
            for dicom in unmatched_dicom:
                dicom_kw = extract_main_keyword(dicom)
                if dicom_kw == rt_kw:
                    found_kw_match = dicom
                    break

        if found_kw_match:
            unmatched_dicom.remove(found_kw_match)
            mappings.append({
                "rtstruct_series": rt,
                "matched_dicom_series": found_kw_match,
                "similarity": 1.0,
                "method": "keyword"
            })
        else:
            still_unmatched_rt.append(rt)

    # Third pass: elimination (pair remaining 1-to-1 if equal count)
    if len(still_unmatched_rt) == len(unmatched_dicom):
        for rt, dicom in zip(still_unmatched_rt, unmatched_dicom):
            mappings.append({
                "rtstruct_series": rt,
                "matched_dicom_series": dicom,
                "similarity": 0.4,  # low confidence
                "method": "elimination"
            })
    else:
        # Anything left unmatched is a true miss
        for rt in still_unmatched_rt:
            unmatched_count += 1
            mappings.append({
                "rtstruct_series": rt,
                "matched_dicom_series": "No match",
                "similarity": 0.0,
                "method": "unmatched"
            })

    folder_mapping[(case_id, study_info)] = mappings

# Print results
for key, mappings in folder_mapping.items():
    case_id, study_info = key
    print(f"\nPatient: {case_id} | Study: {study_info}")
    for m in mappings:
        print(f"{m['rtstruct_series']} -> {m['matched_dicom_series']} "
              f"[similarity={m['similarity']} | method={m['method']}]")

# Summary
print("\n" + "="*50)
print(f"Total RTStruct series processed: {total_rtstruct_count}")
print(f"Total unmatched RTStruct series: {unmatched_count}")
print(f"Matched successfully: {total_rtstruct_count - unmatched_count}")
print(f"Match rate: {100 * (total_rtstruct_count - unmatched_count) / total_rtstruct_count:.2f}%")
print("="*50)


In [ ]:
# =============================
# STEP 11: Save mapping to CSV
# =============================

import pandas as pd

# Flatten all mappings into a single list of rows
mapping_rows = []
for (case_id, study_info), mappings in folder_mapping.items():
    for m in mappings:
        mapping_rows.append({
            "case_id": case_id,
            "study_info": study_info,
            "rtstruct_series": m["rtstruct_series"],
            "matched_dicom_series": m["matched_dicom_series"],
            "similarity": m["similarity"],
            "method": m["method"]
        })

# Create DataFrame
mapping_df = pd.DataFrame(mapping_rows)

# Save CSV
csv_path = "data/rtstruct_dicom_mapping.csv"
mapping_df.to_csv(csv_path, index=False)

print(f"Mapping saved to {csv_path}")
print(f"Total rows saved: {len(mapping_df)}")
